[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Why Documents &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell, which starts MongoDB and seeds it. Run it first. Each
task opens its own client and closes it, so they can be run in any order.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


**1.** One document, nested and with an array.


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

shop.people.delete_many({"handle": "rk"})
shop.people.insert_one({
    "handle": "rk",
    "address": {"city": "Leeds", "postcode": "LS1 4AP"},
    "languages": ["python", "go"],
})

person = shop.people.find_one({"handle": "rk"}, {"_id": 0})
print(person)
print("the city, through the nesting:", person["address"]["city"])
client.close()


{'handle': 'rk', 'address': {'city': 'Leeds', 'postcode': 'LS1 4AP'}, 'languages': ['python', 'go']}
the city, through the nesting: Leeds


The `{"_id": 0}` is there because the `_id` MongoDB invented is different every time this runs, so
printing it would make the output unrepeatable. **BSON Types** is where that matters for real.


**2.** What the seed left behind.


In [3]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

print("products:", shop.products.count_documents({}))
print("reviews: ", shop.reviews.count_documents({}))
print("product 7:", shop.products.find_one({"_id": 7}))
client.close()


products: 500
reviews:  767
product 7: {'_id': 7, 'sku': 'KEY-000007', 'name': 'Dalgo keyboard 7', 'maker': 'Dalgo', 'kind': 'keyboard', 'price': 1750.8, 'stock': 250, 'tags': ['refurbished', 'sale'], 'size': {'w': 40, 'h': 20}}


Every notebook in this guide starts from exactly these documents, because the boot cell calls
`random.seed(0)` before it builds them.


**3.** Into a subdocument, with a dot.


In [4]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

wide = shop.products.count_documents({"size.w": {"$gt": 50}})
print("wider than 50:", wide)
print("an example:", shop.products.find_one({"size.w": {"$gt": 50}}, {"name": 1, "size": 1}))
client.close()


wider than 50: 94
an example: {'_id': 10, 'name': 'Corvid laptop 10', 'size': {'w': 58, 'h': 35}}


`"size.w"` is a path, not a field name with a dot in it. Writing `{"size": {"w": 51}}` instead would
ask for a document whose `size` is exactly that one key, which is the subject of
**Query Operators**.


**4.** A value in an array, with no operator at all.


In [5]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

print("tagged sale:", shop.products.count_documents({"tags": "sale"}))
print("one of them:", shop.products.find_one({"tags": "sale"}, {"name": 1, "tags": 1}))
client.close()


tagged sale: 220
one of them: {'_id': 0, 'name': 'Aster laptop 0', 'tags': ['bulk', 'sale']}


No `$in`, no `$contains`. A query against an array matches when any element matches, and the same
query would also match a document whose `tags` was the plain string `"sale"`.


**5.** Asking the server what it is.


In [6]:
client = pymongo.MongoClient(URI, tz_aware=True)

hello = client.admin.command("hello")
print("replica set:", hello.get("setName"))
print("primary:   ", hello.get("isWritablePrimary"))
print("version:   ", client.admin.command("buildInfo")["version"].split(".")[0])
client.close()


replica set: rs0
primary:    True
version:    8


`setName` is present only because the boot cell ran `replSetInitiate`. On a plain `mongod` it is
absent, and that is what **Bulk Writes and Transactions** uses to show why transactions refuse to
start.


**6.** Two lines that do not agree on their fields.


In [7]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

shop.orders.delete_many({"reference": "AB-1101"})
shop.orders.insert_one({
    "reference": "AB-1101",
    "lines": [
        {"sku": "LAP-000000", "quantity": 1, "price": 1689.62},
        {"sku": "GIFT-CARD", "quantity": 1, "value": 50.00, "expires": "2027-01-01"},
    ],
})

for line in shop.orders.find_one({"reference": "AB-1101"})["lines"]:
    print(" ", sorted(line))
print("nothing objected, because no column list exists to object with")
client.close()


  ['price', 'quantity', 'sku']
  ['expires', 'quantity', 'sku', 'value']
nothing objected, because no column list exists to object with


Relationally this is a nullable column for every field either kind of line might have, or a second
table. Here it is just two documents in a list. The cost arrives when something reads
`line["price"]` on the gift card, and that is what **Beanie Documents** exists to put back.


---

&#8592; **Back to:** [Why Documents](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/01-why-documents.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
